### Diffuse Irradiation with PVlib


install [pvlib package](https://pvlib-python.readthedocs.io/en/stable/installation.html)


In [ ]:
import sys
import os
import getpass
import pandas as pd
import numpy as np
import json
from sqlalchemy import *
# plot
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objs as go
import plotly.offline as pltly
import plotly.express as px

import colorlover as cl
import seaborn as sns
# notebook
from IPython.display import Image
from IPython.core.display import HTML 

# pvlib
import pvlib

pltly.init_notebook_mode(connected=True)
%matplotlib inline

version = 'v1 (jupyter)'
project = 'pv3'

In [ ]:
# check for csv files in upper directory
for root, dirs, files in os.walk('./'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

### values needed in polysun
- [x] Gh - Globalstrahlung [Wh/m2]
- [ ] Dh  - Diffusstrahlung [Wh/m2]
- [ ] Lh - Langwellenstrahlung [Wh/m2]
- [x] Tamb Umgebungstemperatur [°C]
- [x] Vwnd Wind [m/s]
- [x] Hrel Luftfeuchtigkeit [%]

## import htw weatherdata

In [ ]:
df_htw = pd.read_csv('htw_wetter_weatherdata_2015.csv', encoding='latin1', sep=';', index_col=0, parse_dates=True)#, skiprows=3)
df_htw.head()

### resample

In [ ]:
df_htw_h = df_htw.resample('H').mean()
df_htw_h.head()

In [ ]:
fig = px.line(df_htw_h, title='htw_wetter_weatherdata_2015')

# deactivate all but first column
for i, _ in enumerate(fig.data):
    if i==0: 
        fig.data[i].update({'visible':True})
    else:
        fig.data[i].update({'visible':'legendonly'})
fig.show()

In [ ]:
# HTW coords
lat = 52.455778
lon = 13.523917

### compute solarpositiont / zenith

In [ ]:
df_solarpos = pvlib.solarposition.spa_python(df_htw_h.index, lat, lon)
df_solarpos.head()

In [ ]:
df_irradiance = pvlib.irradiance.erbs(ghi=df_htw_h.loc[:,'G_hor_CMP6'],
                      zenith=df_solarpos.zenith,
                      datetime_or_doy=df_htw_h.index.dayofyear)
df_irradiance = pd.DataFrame(df_irradiance)

In [ ]:
df_irradiance

## Dataformat import